# U4 Saber Hacer: análisis no supervisado (COVID Tracking Project)

Agrupación de los 50 estados y DC según la forma de su curva semanal de casos nuevos (marzo 2020 a febrero 2021). Las funciones están en `src/u4_unsupervised_analysis.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import u4_unsupervised_analysis as A
pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', 12)

## 1. Dataset original y dataset limpio

In [2]:
raw = pd.read_csv(A.RAW_FILE, usecols=['date', 'state', 'positiveIncrease'])
print('Dataset original:', raw.shape)
clean = A.load_clean()
print('Dataset limpio (U3):', clean.shape, '| estados:', clean['state'].nunique())

Dataset original: (20780, 3)
Dataset limpio (U3): (18995, 10) | estados: 51


## 2. Dataset transformado: una fila por estado, 52 semanas en escala 0-1

In [3]:
profiles = A.build_weekly_profiles(clean)
weeks = A.week_columns(profiles)
print('Dataset transformado:', profiles[weeks].shape)
profiles[['total_cases', 'peak_week'] + weeks[:6]].head(8)

Dataset transformado: (51, 52)


,total_cases,peak_week,2020-03-02,2020-03-09,2020-03-16,2020-03-23,2020-03-30,2020-04-06
state,,,,,,,,
AK,55989,2020-11-30,0.000e+00,0.000e+00,0.005,0.017,0.015,0.018
AL,493252,2021-01-04,0.000e+00,4.005e-04,0.004,0.022,0.033,0.058
AR,322815,2021-01-04,0.000e+00,7.407e-04,0.007,0.012,0.019,0.021
AZ,816780,2021-01-04,4.826e-05,1.126e-04,0.002,0.012,0.022,0.020
CA,3475509,2020-12-14,1.156e-04,6.773e-04,0.004,0.014,0.026,0.028
CO,428301,2020-11-16,1.990e-04,3.468e-03,0.013,0.049,0.075,0.067
CT,279973,2020-12-07,5.250e-05,9.976e-04,0.011,0.093,0.193,0.334
DC,40598,2021-01-11,4.854e-04,7.282e-03,0.040,0.118,0.318,0.426


## 3. Reducción de dimensionalidad (PCA) y número de componentes

In [4]:
X = profiles[weeks].to_numpy()
A.search_pca_components(X)

,components,variance,best_k,silhouette
0,2,0.654,2,0.517
1,3,0.743,2,0.456
2,4,0.816,2,0.419
3,5,0.860,2,0.401
4,6,0.900,2,0.380
5,8,0.938,2,0.364
6,10,0.958,2,0.356
7,15,0.981,2,0.346
8,20,0.991,2,0.341
9,52,1.000,2,0.337


## 4. Agrupación, evaluación y optimización

In [5]:
for d in (A.TRANSFORMED_DIR, A.FIG_DIR, A.TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)
result, Z, pca = A.run_clustering(profiles)
print('Componentes PCA:', result.summary['pca_components'],
      f"({result.summary['pca_variance']:.1%} de la varianza)")
result.k_search.pivot(index='k', columns='model', values=['silhouette', 'calinski_harabasz', 'davies_bouldin'])

Componentes PCA: 6 (90.0% de la varianza)


silhouette         calinski_harabasz            davies_bouldin  \
model Jerárquico (Ward) K-Means Jerárquico (Ward) K-Means Jerárquico (Ward)   
k                                                                             
2                 0.355   0.380            30.118  38.115             0.850   
3                 0.221   0.311            24.986  29.578             1.347   
4                 0.261   0.293            25.473  28.185             1.211   
5                 0.262   0.298            24.141  27.229             1.001   
6                 0.263   0.286            23.331  25.598             0.970   
7                 0.247   0.228            21.684  24.809             1.058   
8                 0.216   0.245            20.800  23.891             1.117   

               
model K-Means  
k              
2       0.953  
3       1.296  
4       1.137  
5       0.936  
6       0.947  
7       0.996  
8       1.000

In [6]:
result.dbscan_search.sort_values('silhouette', ascending=False).head(8)

,min_samples,eps,clusters,noise,silhouette,calinski_harabasz,davies_bouldin
36,5,0.499,2,33,0.629,44.728,0.468
1,3,0.379,2,39,0.495,14.355,0.634
20,4,0.462,3,29,0.456,30.768,0.804
2,3,0.399,4,33,0.430,20.394,0.730
21,4,0.469,3,28,0.412,24.276,0.823
3,3,0.410,5,28,0.393,21.698,0.723
38,5,0.534,4,15,0.351,27.422,0.899
6,3,0.458,5,19,0.337,27.347,0.887


## 5. Comparación de modelos y antes vs después de la optimización

In [7]:
result.comparison

,model,params,clusters,noise,silhouette,calinski_harabasz,davies_bouldin,ari_vs_kmeans
0,K-Means,k = 4,4,0,0.293,28.185,1.137,1.000
1,Jerárquico (Ward),k = 4,4,0,0.261,25.473,1.211,0.613
2,DBSCAN,eps = 0.53; min_samples = 5,4,15,0.351,27.422,0.899,0.453


In [8]:
result.before_after

,model,stage,params,clusters,noise,silhouette,calinski_harabasz,davies_bouldin
0,K-Means,Antes,k = 8 (por defecto),8,0,0.148,14.806,1.254
1,K-Means,Después,k = 4,4,0,0.237,21.650,1.331
2,Jerárquico (Ward),Antes,k = 2 (por defecto),2,0,0.315,25.584,0.972
3,Jerárquico (Ward),Después,k = 4,4,0,0.211,19.929,1.398
4,DBSCAN,Antes,eps = 0.5; min_samples = 5 (por defecto),0,51,NaN,NaN,NaN
5,DBSCAN,Después,eps = 0.53; min_samples = 5,4,15,0.280,19.600,1.101


## 6. Grupos encontrados

In [9]:
clusters = A.describe_clusters(profiles, result.labels)
clusters

,cluster,states,members,peak_week,share_mar_may,share_jun_ago,share_sep_nov,share_dic_feb
0,1,14,"AK, CO, IA, IL, MI, MN, MO, MT, ND, NE, NM, SD...",2020-11-16,0.048,0.106,0.535,0.311
1,2,14,"ID, IN, KS, KY, NV, OH, OK, OR, PA, RI, TN, UT...",2020-12-07,0.040,0.128,0.367,0.465
2,3,10,"CT, DC, DE, MA, MD, ME, NH, NJ, NY, VT",2021-01-04,0.148,0.076,0.232,0.544
3,4,13,"AL, AR, AZ, CA, FL, GA, HI, LA, MS, NC, SC, TX...",2021-01-04,0.040,0.229,0.247,0.483


## 7. Resultados completos

Genera los CSV de `data/transformed`, las tablas de `outputs/u4/tables`, `outputs/u4/results.json` y las gráficas en `outputs/u4/figures` (las gráficas no se suben al repositorio).

In [10]:
A.main()

{
  "dataset": {
    "raw_rows": 20780,
    "clean_rows": 18995,
    "states": 51,
    "weeks": 52,
    "period": [
      "2020-03-02",
      "2021-02-28"
    ]
  },
  "pca_components": 6,
  "pca_variance": 0.900404186324131,
  "pca_variance_by_component": [
    0.495460735001206,
    0.1586489375077004,
    0.08902979602150157,
    0.07259346258891465,
    0.04389404120723852,
    0.04077721399756982
  ],
  "best_k": 4,
  "silhouette_best_k": 2,
  "dbscan": {
    "eps": 0.534,
    "min_samples": 5
  },
  "cluster_sizes": {
    "1": 14,
    "2": 14,
    "3": 10,
    "4": 13
  },
  "before_after": [
    {
      "model": "K-Means",
      "stage": "Antes",
      "params": "k = 8 (por defecto)",
      "clusters": 8,
      "noise": 0,
      "silhouette": 0.14823963263455192,
      "calinski_harabasz": 14.805983981818697,
      "davies_bouldin": 1.253642168108105
    },
    {
      "model": "K-Means",
      "stage": "Después",
      "params": "k = 4",
      "clusters": 4,
      "noise": 0,
 